# Regeneración completa de ocurrencias BERT

Operación costosa, desactivada por defecto. Usa los vocabularios incluidos y escribe en SalidasRegeneracion/Ocurrencias. No se ha ejecutado esta regeneración completa durante la integración. Se conserva el algoritmo del notebook recibido, con rutas portables y una secuencia única.

Las revisiones no acreditadas permanecen explícitamente sin fijar; no se garantiza identidad numérica al cambiar el entorno BERT. Las huellas de selección y de prototipos deben coincidir antes de aceptar una nueva salida.


In [1]:
from pathlib import Path
import sys

def localizar_base():
    for parent in (Path.cwd(), *Path.cwd().parents):
        for candidate in (parent, parent / 'EstudioContextual/GeneracionDataset', parent / 'GeneracionDataset'):
            if (candidate / 'Codigo/contextual.py').is_file():
                return candidate.resolve()
    raise FileNotFoundError('Abre el notebook desde ProyectoTesis o una de sus subcarpetas.')

BASE = localizar_base()
sys.path.insert(0, str(BASE / 'Codigo'))
import contextual as ctx
print('Carpeta:', BASE)
GENERAR_DESDE_CORPUS = False
MODEL_REVISION = None  # La revisión histórica no consta en los archivos recibidos.
DATASET_REVISIONS = {
    'wikitext': None,
    'wikimedia/wikipedia': 'b04c8d1ceb2f5cd4588862100d08de323dccfbaa',
}
if not GENERAR_DESDE_CORPUS:
    print('Regeneración desactivada. Usa GeneracionDataset.ipynb para reutilizar.')


Carpeta: /workspace/scratch/e285fbe25f1c/figuras_manuscrito/ProyectoTesis/EstudioContextual/GeneracionDataset
Regeneración desactivada. Usa GeneracionDataset.ipynb para reutilizar.


## Dependencias explícitas


In [2]:
if GENERAR_DESDE_CORPUS:
    import importlib.util
    required = ['numpy', 'pandas', 'torch', 'datasets', 'transformers', 'pyarrow', 'tqdm', 'duckdb', 'sklearn']
    missing = [p for p in required if importlib.util.find_spec(p) is None]
    if missing:
        raise RuntimeError('Instala requirements-generacion.txt en el entorno: ' + str(missing))


## Imports y configuración del dispositivo


In [3]:
if GENERAR_DESDE_CORPUS:
    import gc
    import hashlib
    import heapq
    import json
    import math
    import os
    import random
    import re
    import shutil
    import time
    import unicodedata
    from collections import Counter
    from pathlib import Path
    from typing import List
    import duckdb
    import numpy as np
    import pandas as pd
    import pyarrow as pa
    import pyarrow.parquet as pq
    import torch
    from datasets import load_dataset
    from IPython.display import display
    from tqdm.auto import tqdm
    from transformers import AutoModel, AutoTokenizer
    SEED = 42
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
    CPU_LOGICAL_CORES = os.cpu_count() or 8
    CPU_THREADS_DATASET = max(1, min(8, CPU_LOGICAL_CORES - 2))
    os.environ['OMP_NUM_THREADS'] = str(CPU_THREADS_DATASET)
    os.environ['MKL_NUM_THREADS'] = str(CPU_THREADS_DATASET)
    os.environ['OPENBLAS_NUM_THREADS'] = str(CPU_THREADS_DATASET)
    os.environ['NUMEXPR_NUM_THREADS'] = str(CPU_THREADS_DATASET)
    os.environ['RAYON_NUM_THREADS'] = str(CPU_THREADS_DATASET)
    os.environ['TOKENIZERS_PARALLELISM'] = 'true'
    torch.set_num_threads(CPU_THREADS_DATASET)
    try:
        torch.set_num_interop_threads(max(1, min(4, CPU_THREADS_DATASET)))
    except RuntimeError:
        pass
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    if DEVICE.type == 'cuda':
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        try:
            torch.set_float32_matmul_precision('high')
        except Exception:
            pass
    print('CPU_LOGICAL_CORES:', CPU_LOGICAL_CORES)
    print('CPU_THREADS_DATASET:', CPU_THREADS_DATASET)
    print('DEVICE:', DEVICE)
    if DEVICE.type == 'cuda':
        print('GPU:', torch.cuda.get_device_name(0))
        print('VRAM total [GB]:', round(torch.cuda.get_device_properties(0).total_memory / 1024 ** 3, 2))


## Rutas de regeneración y parámetros


In [4]:
if GENERAR_DESDE_CORPUS:
    PROJECT_ROOT = BASE
    OUT_DIR = BASE / 'SalidasRegeneracion/Ocurrencias'
    if str(PROJECT_ROOT) not in sys.path:
        sys.path.insert(0, str(PROJECT_ROOT))
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    CORPUS_AUDIT_ROOT = BASE / 'Entradas'
    AUDITED_ELIGIBLE_VOCAB_PATH = CORPUS_AUDIT_ROOT / 'common_pool_eligible_vocabulary_min100.csv'
    FINAL_ELIGIBLE_VOCAB_PATH = OUT_DIR / 'eligible_vocabulary_min100.csv'
    SELECTED_HASHES_PATH = OUT_DIR / 'selected_context_hashes_top100.parquet'
    SELECTION_REPORT_PATH = OUT_DIR / 'selection_report.json'
    UNSORTED_CANDIDATES_PATH = OUT_DIR / '_contextual_occurrence_candidates_unsorted.parquet'
    CANDIDATES_PATH = OUT_DIR / 'contextual_occurrence_candidates_sorted.parquet'
    DUCKDB_TEMP_DIR = OUT_DIR / '_duckdb_tmp'
    OUT_PARQUET_PATH = OUT_DIR / 'contextual_occurrences_definitive.parquet'
    OUT_PARQUET_PARTIAL_PATH = OUT_DIR / '_contextual_occurrences_definitive.partial.parquet'
    META_PATH = OUT_DIR / 'contextual_dataset_meta.json'
    WORD_COUNTS_PATH = OUT_DIR / 'word_counts.csv'
    CATEGORY_COVERAGE_PATH = OUT_DIR / 'category_words_coverage.csv'
    EXP2_METADATA_COVERAGE_PATH = OUT_DIR / 'exp2_metadata_words_coverage.csv'
    VALIDATION_REPORT_PATH = OUT_DIR / 'validation_report.json'
    SAMPLE_CONTEXTS_PATH = OUT_DIR / 'sample_contexts_by_category_word.csv'
    ROW_GROUP_INDEX_PATH = OUT_DIR / 'word_row_group_index.csv'
    GENERATION_CONFIG_PATH = OUT_DIR / 'generation_config.json'
    MODEL_NAME = 'bert-base-uncased'
    USE_LAYER_AVG = True
    SELECTED_LAYER_IDS = [-4, -3, -2, -1]
    L2_NORMALIZE_EMBEDDINGS = False
    MAX_SEQ_LEN = 128
    WINDOW_TOKENS_EACH_SIDE = 40
    MIN_LINE_WORDS = 4
    MIN_ROW_WORD_LEN = 3
    CONTEXT_KEEP_STOPWORDS = True
    REMOVE_STOPWORDS_FROM_CONTEXT_WINDOW = False
    OCCURRENCES_PER_WORD = 100
    BATCH_SIZE_OCCURRENCES = 64
    FINAL_ROW_GROUP_ROWS = 1000
    CANDIDATE_SORT_ROW_GROUP_ROWS = 100000
    CONTEXTUAL_DATASET_SCHEMA_VERSION = 'ctx_v3_commonpool_equal100_rawbert'
    CONFIG_NAME = 'ctx_v3_bert_base_commonpool_equal100'
    RESET_OUTPUTS = False
    REUSE_SELECTION_MANIFEST = True
    REUSE_SORTED_CANDIDATES = True
    print('OUT_DIR:', OUT_DIR)
    print('AUDITED_ELIGIBLE_VOCAB_PATH:', AUDITED_ELIGIBLE_VOCAB_PATH)
    print('OCCURRENCES_PER_WORD:', OCCURRENCES_PER_WORD)
    print('MODEL_NAME:', MODEL_NAME)
    print('BATCH_SIZE_OCCURRENCES:', BATCH_SIZE_OCCURRENCES)
    print('FINAL_ROW_GROUP_ROWS:', FINAL_ROW_GROUP_ROWS)
    print('RESET_OUTPUTS:', RESET_OUTPUTS)


## Persistencia


In [5]:
if GENERAR_DESDE_CORPUS:
    def json_safe(value):
        if value is None:
            return None
        if isinstance(value, Path):
            return str(value)
        if isinstance(value, (str, bool, int)):
            return value
        if isinstance(value, np.integer):
            return int(value)
        if isinstance(value, np.bool_):
            return bool(value)
        if isinstance(value, (float, np.floating)):
            value = float(value)
            if np.isnan(value):
                return None
            if np.isposinf(value):
                return 'Infinity'
            if np.isneginf(value):
                return '-Infinity'
            return value
        if isinstance(value, pd.DataFrame):
            return [json_safe(row) for row in value.to_dict(orient='records')]
        if isinstance(value, pd.Series):
            return json_safe(value.to_dict())
        if isinstance(value, dict):
            return {str(key): json_safe(item) for key, item in value.items()}
        if isinstance(value, (list, tuple, set, frozenset)):
            return [json_safe(item) for item in value]
        if isinstance(value, np.ndarray):
            return json_safe(value.tolist())
        return str(value)

    def save_json(payload, path):
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        with open(path, 'w', encoding='utf-8') as handle:
            json.dump(json_safe(payload), handle, ensure_ascii=False, indent=2, sort_keys=True)
        return path

    def print_copy_block(title, payload):
        print('\n' + '=' * 90)
        print(f'PARA COPIAR Y PEGAR — {title}')
        print('=' * 90)
        print(json.dumps(json_safe(payload), ensure_ascii=False, indent=2, sort_keys=True))
        print('=' * 90)
        print(f'FIN — {title}')
        print('=' * 90)


## Metadata de las clases


In [6]:
if GENERAR_DESDE_CORPUS:
    category_words = ['apple', 'bicycle', 'bread', 'canoe', 'chair', 'jellyfish', 'screwdriver', 'skunk']
    target_extra_words = ['ant', 'bus', 'cheese', 'crab', 'curtain', 'frog', 'olive', 'parachute', 'peanut', 'pear', 'raccoon', 'raft', 'scooter', 'sling', 'stool', 'wrench']
    target_words = sorted(set(category_words + target_extra_words))
    category_set = set(category_words)
    target_set = set(target_words)
    target_extra_set = set(target_extra_words)
    target_roles = {}
    for word in target_words:
        if word in category_set and word in target_extra_set:
            target_roles[word] = 'category_and_target'
        elif word in category_set:
            target_roles[word] = 'category'
        else:
            target_roles[word] = 'target'
    EXP2_CLASS_LABELS = ['apple', 'bicycle', 'bread', 'canoe', 'chair', 'jellyfish', 'screwdriver', 'skunk']
    EXP2_CLASS_GROUPS = {'apple': ['apple', 'olive', 'pear'], 'bicycle': ['bicycle', 'parachute', 'scooter'], 'bread': ['bread', 'cheese', 'peanut'], 'canoe': ['canoe', 'bus', 'raft'], 'chair': ['chair', 'curtain', 'stool'], 'jellyfish': ['jellyfish', 'ant', 'crab'], 'screwdriver': ['screwdriver', 'sling', 'wrench'], 'skunk': ['skunk', 'frog', 'raccoon']}
    WORD_TO_TRUE_CLASS = {}
    WORD_TO_CLASS_LABEL = {}
    for class_id, class_label in enumerate(EXP2_CLASS_LABELS):
        for word in EXP2_CLASS_GROUPS[class_label]:
            WORD_TO_TRUE_CLASS[word] = int(class_id)
            WORD_TO_CLASS_LABEL[word] = str(class_label)

    def get_exp2_class_metadata(word):
        word = str(word).lower()
        return {'true_class': int(WORD_TO_TRUE_CLASS.get(word, -1)), 'class_label': str(WORD_TO_CLASS_LABEL.get(word, 'filler')), 'is_exp2_word': bool(word in WORD_TO_TRUE_CLASS)}
    if len(category_words) != 8:
        raise RuntimeError(f'Se esperaban 8 CategoryWords y se obtuvieron {len(category_words)}.')
    print('CategoryWords:', category_words)
    print('TargetWords con metadata Exp2:', len(target_words))


## Contrato léxico y hash


In [7]:
if GENERAR_DESDE_CORPUS:
    STRICT_WORD_PATTERN = re.compile('[a-z]+')
    STOPWORDS = set(json.loads((BASE / 'Entradas/stopwords_318.json').read_text(encoding='utf-8')))
    CUSTOM_BAD_WORDS = {'nbsp', 'amp', 'lt', 'gt', 'ref', 'br', 'html', 'http', 'https', 'www', 'com', 'org', 'net', 'url', 'jpg', 'jpeg', 'png', 'gif', 'svg', 'px', 'thumb', 'thumbnail', 'image', 'file', 'align', 'isbn', 'utc', 'doi', 'redirect', 'category', 'template', 'cite', 'citation', 'edit', 'references', 'external', 'links'}
    BAD_WORDS = set(STOPWORDS) | set(CUSTOM_BAD_WORDS)

    def normalize_text(text):
        text = unicodedata.normalize('NFKC', str(text))
        text = text.lower()
        text = text.replace(' @-@ ', '-').replace(' @.@ ', '.').replace(' @,@ ', ',')
        text = re.sub('\\s+', ' ', text).strip()
        return text

    def strict_tokenize(text):
        return STRICT_WORD_PATTERN.findall(normalize_text(text))

    def is_heading_line(text):
        return str(text).lstrip().startswith('=')

    def is_valid_row_word(word):
        word = str(word).lower()
        if len(word) < MIN_ROW_WORD_LEN:
            return False
        if not re.fullmatch('[a-z]+', word):
            return False
        if word in BAD_WORDS:
            return False
        return True

    def stable_doc_split(doc_key, train_p=0.7, val_p=0.15):
        h = hashlib.md5(str(doc_key).encode('utf-8')).hexdigest()
        u = int(h[:8], 16) / 4294967295
        if u < train_p:
            return 'train'
        if u < train_p + val_p:
            return 'validation'
        return 'test'

    def build_centered_window(tokens, word_pos):
        left = max(0, int(word_pos) - WINDOW_TOKENS_EACH_SIDE)
        right = min(len(tokens), int(word_pos) + WINDOW_TOKENS_EACH_SIDE + 1)
        window_tokens = tokens[left:right]
        word_pos_window = int(word_pos) - left
        if REMOVE_STOPWORDS_FROM_CONTEXT_WINDOW:
            cleaned = [token for token in window_tokens if re.fullmatch('[a-z]+', token) and token not in BAD_WORDS]
            word = tokens[int(word_pos)]
            positions = [idx for idx, token in enumerate(cleaned) if token == word]
            if not positions:
                return ([], -1)
            window_tokens = cleaned
            word_pos_window = min(positions, key=lambda idx: abs(idx - len(window_tokens) // 2))
        return (window_tokens, word_pos_window)

    def short_context(window_tokens, word_pos_window, radius=10):
        left = ' '.join(window_tokens[max(0, word_pos_window - radius):word_pos_window])
        right = ' '.join(window_tokens[word_pos_window + 1:word_pos_window + 1 + radius])
        return (left, right)

    def context_priority_int(word, context_window):
        payload = (str(word) + '\x00' + str(context_window)).encode('utf-8')
        digest = hashlib.blake2b(payload, digest_size=16, person=b'ctx100-v1').digest()
        return int.from_bytes(digest, byteorder='big', signed=False)

    def priority_int_to_bytes(value):
        return int(value).to_bytes(16, byteorder='big', signed=False)
    print('Stopwords:', len(STOPWORDS))
    print('CUSTOM_BAD_WORDS:', len(CUSTOM_BAD_WORDS))
    BAD_WORDS.add('nan')


## Vocabulario final congelado


In [8]:
if GENERAR_DESDE_CORPUS:
    eligible_vocab_df = pd.read_csv(BASE / 'Datos/Ocurrencias/eligible_vocabulary_min100.csv', keep_default_na=False, na_filter=False)
    eligible_words = eligible_vocab_df.word.tolist()
    eligible_word_set = set(eligible_words)
    assert len(eligible_words) == 103029 and len(eligible_word_set) == 103029
    assert eligible_words == sorted(eligible_words)
    assert all((is_valid_row_word(w) for w in eligible_words))
    assert set(category_words) <= eligible_word_set
    eligible_vocab_df.to_csv(FINAL_ELIGIBLE_VOCAB_PATH, index=False)


## Corpus común


In [9]:
if GENERAR_DESDE_CORPUS:
    CORPUS_SOURCES = [{'source_name': 'wikitext_103_raw', 'dataset_name': 'wikitext', 'config': 'wikitext-103-raw-v1', 'splits': ['train', 'validation', 'test'], 'text_field': 'text', 'streaming': False, 'max_records_per_split': None}, {'source_name': 'wikimedia_wikipedia_20231101_en_common', 'dataset_name': 'wikimedia/wikipedia', 'config': '20231101.en', 'splits': ['train'], 'text_field': 'text', 'streaming': True, 'max_records_per_split': 800000}]

    def load_hf_split(dataset_name, config, split, streaming):
        if config is None:
            return load_dataset(dataset_name, split=split, streaming=streaming, revision=DATASET_REVISIONS.get(dataset_name))
        return load_dataset(dataset_name, config, split=split, streaming=streaming, revision=DATASET_REVISIONS.get(dataset_name))

    def iter_corpus_texts():
        for source in CORPUS_SOURCES:
            source_name = source['source_name']
            dataset_name = source['dataset_name']
            config = source.get('config')
            text_field = source.get('text_field', 'text')
            streaming = bool(source.get('streaming', False))
            max_records = source.get('max_records_per_split')
            for split in source['splits']:
                print('\nCargando fuente:', source_name, '| split=', split, '| streaming=', streaming)
                try:
                    dataset = load_hf_split(dataset_name, config, split, streaming)
                except Exception as exc:
                    raise RuntimeError(f'No se pudo cargar una fuente que forma parte del pool científico definitivo: {source_name}/{split}. Error: {repr(exc)}') from exc
                for row_idx, row in enumerate(dataset):
                    if max_records is not None and row_idx >= max_records:
                        break
                    text = row.get(text_field, '')
                    if not isinstance(text, str):
                        continue
                    if not text.strip():
                        continue
                    yield {'source_name': source_name, 'source_split': split, 'source_row_idx': int(row_idx), 'text': text}
    display(pd.DataFrame(CORPUS_SOURCES))


## Selección top100


In [10]:
if GENERAR_DESDE_CORPUS:
    HASH_MANIFEST_SCHEMA = pa.schema([('word', pa.string()), ('priority', pa.binary(16))])

    def save_selected_hash_manifest(selected_scores_by_word, path):
        path = Path(path)
        if path.exists():
            path.unlink()
        writer = pq.ParquetWriter(path, schema=HASH_MANIFEST_SCHEMA, compression='zstd', compression_level=3)
        try:
            words_chunk = []
            priorities_chunk = []
            for word in sorted(selected_scores_by_word):
                scores = sorted(selected_scores_by_word[word])
                for score in scores:
                    words_chunk.append(word)
                    priorities_chunk.append(priority_int_to_bytes(score))
                if len(words_chunk) >= 100000:
                    table = pa.Table.from_arrays([pa.array(words_chunk, type=pa.string()), pa.array(priorities_chunk, type=pa.binary(16))], schema=HASH_MANIFEST_SCHEMA)
                    writer.write_table(table, row_group_size=100000)
                    words_chunk.clear()
                    priorities_chunk.clear()
            if words_chunk:
                table = pa.Table.from_arrays([pa.array(words_chunk, type=pa.string()), pa.array(priorities_chunk, type=pa.binary(16))], schema=HASH_MANIFEST_SCHEMA)
                writer.write_table(table, row_group_size=100000)
        finally:
            writer.close()

    def load_selected_hash_manifest(path, eligible_words):
        selected = {word: set() for word in eligible_words}
        parquet_file = pq.ParquetFile(path)
        for batch in tqdm(parquet_file.iter_batches(batch_size=200000, columns=['word', 'priority']), desc='Cargando manifest de hashes'):
            words_batch = batch.column(0).to_pylist()
            priorities_batch = batch.column(1).to_pylist()
            for word, priority_bytes in zip(words_batch, priorities_batch):
                if word not in selected:
                    raise RuntimeError(f'El manifest contiene una palabra fuera del vocabulario definitivo: {word}')
                selected[word].add(int.from_bytes(priority_bytes, byteorder='big', signed=False))
        bad_counts = {word: len(scores) for word, scores in selected.items() if len(scores) != OCCURRENCES_PER_WORD}
        if bad_counts:
            raise RuntimeError(f'Manifest de selección inválido. Ejemplos: {list(bad_counts.items())[:20]}')
        return selected

    def compute_top100_context_hashes():
        heaps = {word: [] for word in eligible_words}
        selected = {word: set() for word in eligible_words}
        source_documents = Counter()
        source_valid_occurrences = Counter()
        n_words_total = int(len(eligible_words))
        n_slots_total = int(n_words_total * OCCURRENCES_PER_WORD)
        selected_slots_total = 0
        heap_full_count = 0
        global_documents_seen = 0
        started = time.perf_counter()
        for source in CORPUS_SOURCES:
            source_name = source['source_name']
            dataset_name = source['dataset_name']
            config = source.get('config')
            text_field = source.get('text_field', 'text')
            streaming = bool(source.get('streaming', False))
            max_records = source.get('max_records_per_split')
            for split in source['splits']:
                print('\n' + '=' * 90)
                print('SELECCIÓN TOP-100')
                print('Fuente:', source_name)
                print('Split:', split)
                print('Streaming:', streaming)
                print('=' * 90)
                try:
                    dataset = load_hf_split(dataset_name, config, split, streaming)
                except Exception as exc:
                    raise RuntimeError(f'No se pudo cargar una fuente perteneciente al pool científico definitivo: {source_name}/{split}. Error={repr(exc)}') from exc
                if max_records is not None:
                    source_total = int(max_records)
                elif not streaming and hasattr(dataset, '__len__'):
                    source_total = int(len(dataset))
                else:
                    source_total = None
                source_desc = f'{source_name}/{split}'
                pbar = tqdm(total=source_total, desc=source_desc, unit='docs', dynamic_ncols=True)
                try:
                    for row_idx, row in enumerate(dataset):
                        if max_records is not None and row_idx >= max_records:
                            break
                        pbar.update(1)
                        global_documents_seen += 1
                        source_documents[source_name] += 1
                        text = row.get(text_field, '')
                        if not isinstance(text, str) or not text.strip():
                            continue
                        if is_heading_line(text):
                            continue
                        tokens = strict_tokenize(text)
                        if len(tokens) < MIN_LINE_WORDS:
                            continue
                        for position, word in enumerate(tokens):
                            if word not in eligible_word_set:
                                continue
                            window_tokens, word_pos_window = build_centered_window(tokens, position)
                            if not window_tokens or word_pos_window < 0 or word_pos_window >= len(window_tokens):
                                continue
                            if window_tokens[word_pos_window] != word:
                                continue
                            if len(window_tokens) < MIN_LINE_WORDS:
                                continue
                            context_window = ' '.join(window_tokens)
                            score = context_priority_int(word, context_window)
                            score_set = selected[word]
                            heap = heaps[word]
                            if score in score_set:
                                continue
                            source_valid_occurrences[source_name] += 1
                            if len(heap) < OCCURRENCES_PER_WORD:
                                heapq.heappush(heap, -score)
                                score_set.add(score)
                                selected_slots_total += 1
                                if len(heap) == OCCURRENCES_PER_WORD:
                                    heap_full_count += 1
                            else:
                                current_largest_selected = -heap[0]
                                if score < current_largest_selected:
                                    old_negative = heapq.heapreplace(heap, -score)
                                    old_score = -old_negative
                                    score_set.remove(old_score)
                                    score_set.add(score)
                        if pbar.n % 2000 == 0:
                            pbar.set_postfix({'heap_full': f'{heap_full_count:,}/{n_words_total:,}', 'slots': f'{selected_slots_total:,}/{n_slots_total:,}', 'global_docs': f'{global_documents_seen:,}'}, refresh=False)
                finally:
                    pbar.close()
                print('\nFinalizado:', source_desc)
                print('Documentos procesados en la fuente:', source_documents[source_name])
                print('Palabras con heap lleno:', f'{heap_full_count:,}/{n_words_total:,}')
                print('Slots ocupados:', f'{selected_slots_total:,}/{n_slots_total:,}')
        elapsed_seconds = float(time.perf_counter() - started)
        bad_counts = {word: len(scores) for word, scores in selected.items() if len(scores) != OCCURRENCES_PER_WORD}
        if bad_counts:
            raise RuntimeError(f'El pool común no produjo exactamente {OCCURRENCES_PER_WORD} contextos seleccionables para todas las palabras auditadas. N problemáticas={len(bad_counts)}. Ejemplos={list(bad_counts.items())[:20]}')
        if heap_full_count != n_words_total:
            raise RuntimeError('Inconsistencia interna: no todas las palabras terminaron con el heap completo.')
        if selected_slots_total != n_slots_total:
            raise RuntimeError('Inconsistencia interna en el número de slots seleccionados.')
        selection_hasher = hashlib.sha256()
        for word in sorted(selected):
            selection_hasher.update(word.encode('utf-8'))
            selection_hasher.update(b'\x00')
            for score in sorted(selected[word]):
                selection_hasher.update(priority_int_to_bytes(score))
        report = {'selection_method': 'blake2b_128_keep_100_smallest_unique_word_context_hashes', 'selection_personalization': 'ctx100-v1', 'occurrences_per_word': int(OCCURRENCES_PER_WORD), 'n_words': int(n_words_total), 'n_selected_contexts': int(n_slots_total), 'all_heaps_full': bool(heap_full_count == n_words_total), 'selection_sha256': selection_hasher.hexdigest(), 'source_documents_scanned': source_documents, 'source_valid_occurrences_seen': source_valid_occurrences, 'global_documents_scanned': int(global_documents_seen), 'elapsed_seconds': elapsed_seconds}
        return (selected, report)
    if REUSE_SELECTION_MANIFEST and SELECTED_HASHES_PATH.exists():
        print('Reutilizando manifest existente:', SELECTED_HASHES_PATH)
        selected_hashes_by_word = load_selected_hash_manifest(SELECTED_HASHES_PATH, eligible_words)
        if SELECTION_REPORT_PATH.exists():
            with open(SELECTION_REPORT_PATH, 'r', encoding='utf-8') as handle:
                selection_report = json.load(handle)
        else:
            selection_report = {'selection_method': 'loaded_existing_manifest', 'occurrences_per_word': OCCURRENCES_PER_WORD, 'n_words': len(eligible_words), 'n_selected_contexts': len(eligible_words) * OCCURRENCES_PER_WORD}
    else:
        selected_hashes_by_word, selection_report = compute_top100_context_hashes()
        save_selected_hash_manifest(selected_hashes_by_word, SELECTED_HASHES_PATH)
        save_json(selection_report, SELECTION_REPORT_PATH)
    print_copy_block('SELECCIÓN HASH TOP100', selection_report)


## Verificar selección antes de BERT


In [11]:
if GENERAR_DESDE_CORPUS:
    h = hashlib.sha256()
    for word in sorted(selected_hashes_by_word):
        h.update(word.encode('utf-8'))
        h.update(b'\x00')
        for score in sorted(selected_hashes_by_word[word]):
            h.update(priority_int_to_bytes(score))
    if h.hexdigest() != ctx.SELECTION_SHA:
        raise RuntimeError('La selección de contextos no coincide con el dataset definitivo: ' + h.hexdigest())
    selection_report['selection_sha256'] = h.hexdigest()
    save_json(selection_report, SELECTION_REPORT_PATH)


## Materializar contextos


In [12]:
if GENERAR_DESDE_CORPUS:
    CANDIDATE_SCHEMA = pa.schema([('candidate_id', pa.int64()), ('word', pa.string()), ('is_target', pa.bool_()), ('word_role', pa.string()), ('true_class', pa.int32()), ('class_label', pa.string()), ('is_exp2_word', pa.bool_()), ('selection_stage', pa.string()), ('sample_priority', pa.binary(16)), ('source_name', pa.string()), ('source_split', pa.string()), ('source_row_idx', pa.int64()), ('source_doc_key', pa.string()), ('doc_split', pa.string()), ('word_pos_original', pa.int32()), ('word_pos_window', pa.int32()), ('n_tokens_window', pa.int32()), ('context_window', pa.string()), ('left_context', pa.string()), ('right_context', pa.string()), ('text', pa.string())])

    def records_to_candidate_table(records):
        arrays = []
        for field in CANDIDATE_SCHEMA:
            arrays.append(pa.array([row[field.name] for row in records], type=field.type))
        return pa.Table.from_arrays(arrays, schema=CANDIDATE_SCHEMA)

    def make_candidate_record(*, candidate_id, word, score, source_name, source_split, source_row_idx, text, tokens, word_pos_original):
        window_tokens, word_pos_window = build_centered_window(tokens, word_pos_original)
        if not window_tokens or word_pos_window < 0 or word_pos_window >= len(window_tokens):
            return None
        if window_tokens[word_pos_window] != word:
            return None
        if len(window_tokens) < MIN_LINE_WORDS:
            return None
        context_window = ' '.join(window_tokens)
        score_check = context_priority_int(word, context_window)
        if score_check != score:
            raise RuntimeError('La prioridad recalculada no coincide con la selección congelada.')
        left_context, right_context = short_context(window_tokens, word_pos_window)
        source_doc_key = f'{source_name}:{source_split}:{source_row_idx}'
        class_meta = get_exp2_class_metadata(word)
        return {'candidate_id': int(candidate_id), 'word': str(word), 'is_target': bool(word in target_set), 'word_role': str(target_roles.get(word, 'filler')), 'true_class': int(class_meta['true_class']), 'class_label': str(class_meta['class_label']), 'is_exp2_word': bool(class_meta['is_exp2_word']), 'selection_stage': 'common_pool_hash_top100', 'sample_priority': priority_int_to_bytes(score), 'source_name': str(source_name), 'source_split': str(source_split), 'source_row_idx': int(source_row_idx), 'source_doc_key': str(source_doc_key), 'doc_split': str(stable_doc_split(source_doc_key)), 'word_pos_original': int(word_pos_original), 'word_pos_window': int(word_pos_window), 'n_tokens_window': int(len(window_tokens)), 'context_window': context_window, 'left_context': left_context, 'right_context': right_context, 'text': normalize_text(text)[:1000]}

    def materialize_selected_candidates(selected_hashes_by_word):
        if UNSORTED_CANDIDATES_PATH.exists():
            UNSORTED_CANDIDATES_PATH.unlink()
        remaining = {word: set(scores) for word, scores in selected_hashes_by_word.items()}
        remaining_total = int(len(eligible_words) * OCCURRENCES_PER_WORD)
        candidate_id = 0
        source_written = Counter()
        buffer = []
        writer = pq.ParquetWriter(UNSORTED_CANDIDATES_PATH, schema=CANDIDATE_SCHEMA, compression='zstd', compression_level=3)
        started = time.perf_counter()
        pbar = tqdm(total=remaining_total, desc='Materializando candidatos top100', unit='ctx')
        try:
            for item in iter_corpus_texts():
                if remaining_total == 0:
                    break
                text = item['text']
                if is_heading_line(text):
                    continue
                tokens = strict_tokenize(text)
                if len(tokens) < MIN_LINE_WORDS:
                    continue
                source_name = item['source_name']
                source_split = item['source_split']
                source_row_idx = item['source_row_idx']
                for position, word in enumerate(tokens):
                    scores_for_word = remaining.get(word)
                    if not scores_for_word:
                        continue
                    window_tokens, word_pos_window = build_centered_window(tokens, position)
                    if not window_tokens or word_pos_window < 0 or word_pos_window >= len(window_tokens):
                        continue
                    if window_tokens[word_pos_window] != word:
                        continue
                    context_window = ' '.join(window_tokens)
                    score = context_priority_int(word, context_window)
                    if score not in scores_for_word:
                        continue
                    record = make_candidate_record(candidate_id=candidate_id, word=word, score=score, source_name=source_name, source_split=source_split, source_row_idx=source_row_idx, text=text, tokens=tokens, word_pos_original=position)
                    if record is None:
                        raise RuntimeError('Un contexto seleccionado no pudo materializarse.')
                    buffer.append(record)
                    scores_for_word.remove(score)
                    candidate_id += 1
                    remaining_total -= 1
                    source_written[source_name] += 1
                    pbar.update(1)
                    if len(buffer) >= 50000:
                        writer.write_table(records_to_candidate_table(buffer), row_group_size=50000)
                        buffer.clear()
                if remaining_total == 0:
                    break
            if buffer:
                writer.write_table(records_to_candidate_table(buffer), row_group_size=50000)
                buffer.clear()
        finally:
            writer.close()
            pbar.close()
        missing = {word: len(scores) for word, scores in remaining.items() if scores}
        if missing:
            raise RuntimeError(f'No fue posible rematerializar todos los contextos seleccionados. N palabras incompletas={len(missing)}. Ejemplos={list(missing.items())[:20]}')
        parquet_file = pq.ParquetFile(UNSORTED_CANDIDATES_PATH)
        expected_rows = int(len(eligible_words) * OCCURRENCES_PER_WORD)
        if parquet_file.metadata.num_rows != expected_rows:
            raise RuntimeError(f'Número incorrecto de candidatos materializados: {parquet_file.metadata.num_rows} != {expected_rows}')
        return {'n_candidates': int(parquet_file.metadata.num_rows), 'n_words': int(len(eligible_words)), 'occurrences_per_word': int(OCCURRENCES_PER_WORD), 'source_written_counts': source_written, 'elapsed_seconds': float(time.perf_counter() - started), 'unsorted_candidate_path': str(UNSORTED_CANDIDATES_PATH)}

    def candidate_file_is_structurally_valid(path):
        path = Path(path)
        if not path.exists():
            return False
        try:
            parquet_file = pq.ParquetFile(path)
            expected_rows = len(eligible_words) * OCCURRENCES_PER_WORD
            return parquet_file.metadata.num_rows == expected_rows and set(CANDIDATE_SCHEMA.names).issubset(parquet_file.schema_arrow.names)
        except Exception:
            return False
    if REUSE_SORTED_CANDIDATES and candidate_file_is_structurally_valid(CANDIDATES_PATH):
        print('Candidatos ordenados existentes y estructuralmente válidos. Se omite rematerialización.')
        candidate_materialization_report = {'reused_sorted_candidates': True, 'candidate_path': str(CANDIDATES_PATH), 'n_candidates': int(pq.ParquetFile(CANDIDATES_PATH).metadata.num_rows)}
    else:
        candidate_materialization_report = materialize_selected_candidates(selected_hashes_by_word)
    print_copy_block('CANDIDATOS MATERIALIZADOS', candidate_materialization_report)


## Ordenar candidatos


In [13]:
if GENERAR_DESDE_CORPUS:
    def duckdb_path(path):
        return str(Path(path).resolve()).replace('\\', '/').replace("'", "''")

    def validate_sorted_candidate_counts(path):
        path_sql = duckdb_path(path)
        con = duckdb.connect(database=':memory:')
        try:
            audit_df = con.execute(f"\n            SELECT\n                COUNT(*) AS n_words,\n                MIN(n) AS min_count,\n                MAX(n) AS max_count,\n                SUM(n) AS n_rows\n            FROM (\n                SELECT\n                    word,\n                    COUNT(*) AS n\n                FROM read_parquet('{path_sql}')\n                GROUP BY word\n            )\n            ").fetchdf()
        finally:
            con.close()
        row = audit_df.iloc[0]
        return {'n_words': int(row['n_words']), 'min_count': int(row['min_count']), 'max_count': int(row['max_count']), 'n_rows': int(row['n_rows'])}
    needs_sort = not candidate_file_is_structurally_valid(CANDIDATES_PATH)
    if needs_sort:
        if not UNSORTED_CANDIDATES_PATH.exists():
            raise FileNotFoundError('No existe el Parquet de candidatos sin ordenar.')
        if CANDIDATES_PATH.exists():
            CANDIDATES_PATH.unlink()
        DUCKDB_TEMP_DIR.mkdir(parents=True, exist_ok=True)
        con = duckdb.connect(database=':memory:')
        try:
            con.execute(f"PRAGMA temp_directory='{duckdb_path(DUCKDB_TEMP_DIR)}'")
            con.execute("SET memory_limit='10GB'")
            con.execute('SET preserve_insertion_order=false')
            source_path_sql = duckdb_path(UNSORTED_CANDIDATES_PATH)
            target_path_sql = duckdb_path(CANDIDATES_PATH)
            query = f"\n        COPY (\n            SELECT *\n            FROM read_parquet('{source_path_sql}')\n            ORDER BY\n                word ASC,\n                sample_priority ASC\n        )\n        TO '{target_path_sql}'\n        (\n            FORMAT PARQUET,\n            COMPRESSION ZSTD,\n            ROW_GROUP_SIZE {int(CANDIDATE_SORT_ROW_GROUP_ROWS)}\n        )\n        "
            print('Ordenando candidatos externamente con DuckDB...')
            started_sort = time.perf_counter()
            con.execute(query)
            sort_elapsed_seconds = float(time.perf_counter() - started_sort)
        finally:
            con.close()
        if not candidate_file_is_structurally_valid(CANDIDATES_PATH):
            raise RuntimeError('El archivo de candidatos ordenado no pasó la validación estructural.')
        candidate_count_audit = validate_sorted_candidate_counts(CANDIDATES_PATH)
        if candidate_count_audit['n_words'] != len(eligible_words):
            raise RuntimeError('Número de palabras incorrecto en candidatos ordenados.')
        if candidate_count_audit['min_count'] != OCCURRENCES_PER_WORD or candidate_count_audit['max_count'] != OCCURRENCES_PER_WORD:
            raise RuntimeError('Los candidatos ordenados no tienen exactamente 100 ocurrencias por palabra.')
        UNSORTED_CANDIDATES_PATH.unlink()
        if DUCKDB_TEMP_DIR.exists():
            try:
                shutil.rmtree(DUCKDB_TEMP_DIR)
            except Exception:
                pass
        candidate_sort_report = {'sorted_candidate_path': str(CANDIDATES_PATH), 'sort_order': ['word ASC', 'sample_priority ASC'], **candidate_count_audit, 'elapsed_seconds': sort_elapsed_seconds}
    else:
        candidate_count_audit = validate_sorted_candidate_counts(CANDIDATES_PATH)
        if candidate_count_audit['n_words'] != len(eligible_words) or candidate_count_audit['min_count'] != OCCURRENCES_PER_WORD or candidate_count_audit['max_count'] != OCCURRENCES_PER_WORD:
            raise RuntimeError('El candidate parquet reutilizado no cumple el contrato exacto de 100 ocurrencias por palabra.')
        candidate_sort_report = {'reused_existing_sorted_candidates': True, 'sorted_candidate_path': str(CANDIDATES_PATH), **candidate_count_audit}
    print_copy_block('CANDIDATOS ORDENADOS', candidate_sort_report)
    if 'selected_hashes_by_word' in globals():
        del selected_hashes_by_word
        gc.collect()


## Comprobar espacio


In [14]:
if GENERAR_DESDE_CORPUS:
    expected_rows = int(len(eligible_words) * OCCURRENCES_PER_WORD)
    raw_embedding_payload_gib = float(expected_rows * 768 * 4 / 1024 ** 3)
    disk_usage = shutil.disk_usage(OUT_DIR)
    free_gib = float(disk_usage.free / 1024 ** 3)
    candidate_size_gib = float(CANDIDATES_PATH.stat().st_size / 1024 ** 3)
    print('Filas esperadas:', expected_rows)
    print('Payload mínimo solo embeddings float32 [GiB]:', round(raw_embedding_payload_gib, 2))
    print('Tamaño candidates [GiB]:', round(candidate_size_gib, 2))
    print('Espacio libre actual [GiB]:', round(free_gib, 2))
    if free_gib < raw_embedding_payload_gib * 1.15:
        raise RuntimeError('El espacio libre es demasiado cercano al payload mínimo de embeddings. Libera espacio antes de iniciar BERT.')


## Cargar BERT


In [15]:
if GENERAR_DESDE_CORPUS:
    if DEVICE.type != 'cuda':
        print('ADVERTENCIA: CUDA no está disponible. La generación completa con BERT en CPU será muy costosa.')
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True, revision=MODEL_REVISION)
    if not tokenizer.is_fast:
        raise RuntimeError('Se necesita un tokenizer fast para usar word_ids().')
    model = AutoModel.from_pretrained(MODEL_NAME, output_hidden_states=False, revision=MODEL_REVISION)
    model.to(DEVICE)
    model.eval()
    hidden_size = int(model.config.hidden_size)
    num_hidden_layers = int(model.config.num_hidden_layers)
    if hidden_size != 768:
        raise RuntimeError(f'Se esperaba BERT-base 768D y se obtuvo {hidden_size}.')
    resolved_layer_indices = [num_hidden_layers + layer_id if layer_id < 0 else layer_id for layer_id in SELECTED_LAYER_IDS]
    if resolved_layer_indices != [8, 9, 10, 11]:
        raise RuntimeError(f'Las capas resueltas no corresponden a las últimas 4 capas de BERT-base: {resolved_layer_indices}')
    captured_layer_outputs = {}

    def _make_capture_hook(layer_idx):

        def hook(module, inputs, output):
            if isinstance(output, tuple):
                tensor = output[0]
            else:
                tensor = output
            captured_layer_outputs[layer_idx] = tensor
        return hook
    capture_hooks = []
    for layer_idx in resolved_layer_indices:
        handle = model.encoder.layer[layer_idx].register_forward_hook(_make_capture_hook(layer_idx))
        capture_hooks.append(handle)
    print('Modelo:', MODEL_NAME)
    print('hidden_size:', hidden_size)
    print('num_hidden_layers:', num_hidden_layers)
    print('Capas capturadas [0-based]:', resolved_layer_indices)
    print('DEVICE:', DEVICE)


## Auditar WordPieces


In [16]:
if GENERAR_DESDE_CORPUS:
    TOKENIZER_ALIGNMENT_AUDIT_PATH = OUT_DIR / 'tokenizer_alignment_audit.json'
    TOKENIZER_AUDIT_BATCH_SIZE = 1024
    N_SPECIAL_TOKENS = int(tokenizer.num_special_tokens_to_add(pair=False))
    MAX_CONTENT_WORDPIECES = int(MAX_SEQ_LEN - N_SPECIAL_TOKENS)
    if N_SPECIAL_TOKENS != 2:
        raise RuntimeError(f'Para bert-base-uncased se esperaban exactamente 2 tokens especiales, recibidos={N_SPECIAL_TOKENS}.')
    if MAX_CONTENT_WORDPIECES != 126:
        raise RuntimeError(f'Con MAX_SEQ_LEN=128 se esperaban 126 WordPieces de contenido, recibidos={MAX_CONTENT_WORDPIECES}.')

    def compute_target_centered_wordpiece_crop(input_ids_no_special, word_ids, target_word_position):
        """
        Calcula un recorte WordPiece centrado en la palabra objetivo.

        Retorna:
        - cropped_ids SIN tokens especiales;
        - posiciones del objetivo en el input final CON [CLS];
        - metadata diagnóstica.

        Nunca corta los WordPieces de la palabra objetivo.
        """
        input_ids_no_special = list(input_ids_no_special)
        word_ids = list(word_ids)
        target_word_position = int(target_word_position)
        target_positions = [idx for idx, word_id in enumerate(word_ids) if word_id == target_word_position]
        if not target_positions:
            raise RuntimeError('La palabra objetivo no tiene WordPieces antes del recorte.')
        target_start = int(min(target_positions))
        target_end = int(max(target_positions))
        expected_target_positions = list(range(target_start, target_end + 1))
        if target_positions != expected_target_positions:
            raise RuntimeError('Los WordPieces de la palabra objetivo no son contiguos.')
        target_len = int(target_end - target_start + 1)
        if target_len > MAX_CONTENT_WORDPIECES:
            raise RuntimeError(f'Una única palabra requiere más WordPieces que todo el presupuesto de BERT: {target_len} > {MAX_CONTENT_WORDPIECES}')
        n_raw_wordpieces = int(len(input_ids_no_special))
        if n_raw_wordpieces <= MAX_CONTENT_WORDPIECES:
            crop_start = 0
            crop_end = n_raw_wordpieces
        else:
            remaining_budget = int(MAX_CONTENT_WORDPIECES - target_len)
            left_available = int(target_start)
            right_available = int(n_raw_wordpieces - target_end - 1)
            desired_left = int(remaining_budget // 2)
            desired_right = int(remaining_budget - desired_left)
            left_take = min(left_available, desired_left)
            right_take = min(right_available, desired_right)
            remaining = int(remaining_budget - left_take - right_take)
            if remaining > 0:
                extra_left = min(left_available - left_take, remaining)
                left_take += extra_left
                remaining -= extra_left
            if remaining > 0:
                extra_right = min(right_available - right_take, remaining)
                right_take += extra_right
                remaining -= extra_right
            crop_start = int(target_start - left_take)
            crop_end = int(target_end + right_take + 1)
        cropped_ids = input_ids_no_special[crop_start:crop_end]
        wp_start = int(1 + target_start - crop_start)
        wp_end = int(1 + target_end - crop_start)
        final_length = int(len(cropped_ids) + N_SPECIAL_TOKENS)
        if final_length > MAX_SEQ_LEN:
            raise RuntimeError(f'El recorte centrado excede MAX_SEQ_LEN: {final_length} > {MAX_SEQ_LEN}')
        if not 1 <= wp_start <= wp_end < final_length - 1:
            raise RuntimeError('Posiciones WordPiece objetivo inválidas después del recorte.')
        return {'cropped_ids': cropped_ids, 'wp_start': wp_start, 'wp_end': wp_end, 'target_n_wordpieces': target_len, 'raw_n_wordpieces': n_raw_wordpieces, 'cropped_n_wordpieces': int(len(cropped_ids)), 'crop_start': int(crop_start), 'crop_end': int(crop_end), 'was_cropped': bool(n_raw_wordpieces > MAX_CONTENT_WORDPIECES), 'final_length_with_special_tokens': final_length}
    candidate_pf_for_alignment = pq.ParquetFile(CANDIDATES_PATH)
    n_alignment_rows_checked = 0
    n_target_missing_before_crop = 0
    n_target_too_long = 0
    n_rows_requiring_crop = 0
    n_naive_target_fully_lost = 0
    n_naive_target_partially_truncated = 0
    n_centered_crop_failures = 0
    max_raw_wordpieces = 0
    max_target_wordpieces = 0
    failure_examples = []
    started_alignment = time.perf_counter()
    for batch in tqdm(candidate_pf_for_alignment.iter_batches(batch_size=TOKENIZER_AUDIT_BATCH_SIZE, columns=['word', 'word_pos_window', 'context_window']), total=math.ceil(candidate_pf_for_alignment.metadata.num_rows / TOKENIZER_AUDIT_BATCH_SIZE), desc='Auditando recorte WordPiece centrado'):
        batch_df = batch.to_pandas()
        batch_tokens = [str(value).split() for value in batch_df['context_window'].tolist()]
        positions = batch_df['word_pos_window'].astype(int).tolist()
        enc = tokenizer(batch_tokens, is_split_into_words=True, add_special_tokens=False, truncation=False, padding=False, return_attention_mask=False, verbose=False)
        for batch_idx, word_position in enumerate(positions):
            ids = enc['input_ids'][batch_idx]
            word_ids = enc.word_ids(batch_index=batch_idx)
            target_positions = [idx for idx, word_id in enumerate(word_ids) if word_id == word_position]
            if not target_positions:
                n_target_missing_before_crop += 1
                if len(failure_examples) < 20:
                    failure_examples.append({'problem': 'target_missing_before_crop', 'word': str(batch_df.iloc[batch_idx]['word']), 'word_pos_window': int(word_position), 'context_window': str(batch_df.iloc[batch_idx]['context_window'])})
                continue
            target_start = int(min(target_positions))
            target_end = int(max(target_positions))
            target_len = int(target_end - target_start + 1)
            max_raw_wordpieces = max(max_raw_wordpieces, len(ids))
            max_target_wordpieces = max(max_target_wordpieces, target_len)
            if target_len > MAX_CONTENT_WORDPIECES:
                n_target_too_long += 1
                continue
            if len(ids) > MAX_CONTENT_WORDPIECES:
                n_rows_requiring_crop += 1
            if target_start >= MAX_CONTENT_WORDPIECES:
                n_naive_target_fully_lost += 1
            elif target_end >= MAX_CONTENT_WORDPIECES:
                n_naive_target_partially_truncated += 1
            try:
                crop_info = compute_target_centered_wordpiece_crop(input_ids_no_special=ids, word_ids=word_ids, target_word_position=word_position)
                if crop_info['final_length_with_special_tokens'] > MAX_SEQ_LEN:
                    raise RuntimeError('final_length > MAX_SEQ_LEN')
                recovered_target_len = int(crop_info['wp_end'] - crop_info['wp_start'] + 1)
                if recovered_target_len != target_len:
                    raise RuntimeError('El recorte no conservó todos los WordPieces del target.')
            except Exception as exc:
                n_centered_crop_failures += 1
                if len(failure_examples) < 20:
                    failure_examples.append({'problem': repr(exc), 'word': str(batch_df.iloc[batch_idx]['word']), 'word_pos_window': int(word_position), 'context_window': str(batch_df.iloc[batch_idx]['context_window'])})
        n_alignment_rows_checked += int(len(batch_df))
    alignment_audit = {'input_policy': 'target_centered_wordpiece_crop', 'n_rows_checked': int(n_alignment_rows_checked), 'max_seq_len': int(MAX_SEQ_LEN), 'n_special_tokens': int(N_SPECIAL_TOKENS), 'max_content_wordpieces': int(MAX_CONTENT_WORDPIECES), 'n_target_missing_before_crop': int(n_target_missing_before_crop), 'n_target_too_long': int(n_target_too_long), 'n_rows_requiring_wordpiece_crop': int(n_rows_requiring_crop), 'n_naive_target_fully_lost': int(n_naive_target_fully_lost), 'n_naive_target_partially_truncated': int(n_naive_target_partially_truncated), 'n_centered_crop_failures': int(n_centered_crop_failures), 'max_raw_wordpieces': int(max_raw_wordpieces), 'max_target_wordpieces': int(max_target_wordpieces), 'all_rows_alignable': bool(n_target_missing_before_crop == 0 and n_target_too_long == 0 and (n_centered_crop_failures == 0)), 'examples': failure_examples, 'elapsed_seconds': float(time.perf_counter() - started_alignment)}
    save_json(alignment_audit, TOKENIZER_ALIGNMENT_AUDIT_PATH)
    print_copy_block('AUDITORÍA WORDPIECE CENTRADA PRE-BERT', alignment_audit)
    if not alignment_audit['all_rows_alignable']:
        raise RuntimeError('Persisten ocurrencias no alineables incluso con recorte WordPiece centrado. No ejecutes BERT.')


## Schema


In [17]:
if GENERAR_DESDE_CORPUS:
    embedding_type = pa.list_(pa.float32(), list_size=hidden_size)
    FINAL_SCHEMA = pa.schema([('occ_id', pa.int64()), ('candidate_id', pa.int64()), ('word', pa.string()), ('is_target', pa.bool_()), ('word_role', pa.string()), ('true_class', pa.int32()), ('class_label', pa.string()), ('is_exp2_word', pa.bool_()), ('selection_stage', pa.string()), ('source_name', pa.string()), ('source_split', pa.string()), ('source_row_idx', pa.int64()), ('source_doc_key', pa.string()), ('doc_split', pa.string()), ('word_pos_original', pa.int32()), ('word_pos_window', pa.int32()), ('wp_start', pa.int32()), ('wp_end', pa.int32()), ('n_tokens_window', pa.int32()), ('context_window', pa.string()), ('left_context', pa.string()), ('right_context', pa.string()), ('text', pa.string()), ('embedding', embedding_type)])
    print(FINAL_SCHEMA)


## Extraer embeddings


In [18]:
if GENERAR_DESDE_CORPUS:
    FINAL_METADATA_COLUMNS = ['candidate_id', 'word', 'is_target', 'word_role', 'true_class', 'class_label', 'is_exp2_word', 'selection_stage', 'source_name', 'source_split', 'source_row_idx', 'source_doc_key', 'doc_split', 'word_pos_original', 'word_pos_window', 'n_tokens_window', 'context_window', 'left_context', 'right_context', 'text']

    def rows_embeddings_to_final_table(rows, embeddings):
        X = np.asarray(embeddings, dtype=np.float32)
        if X.ndim != 2:
            raise RuntimeError(f'Embeddings con forma inválida: {X.shape}')
        if X.shape[1] != hidden_size:
            raise RuntimeError(f'Dimensión inválida: {X.shape[1]} != {hidden_size}')
        arrays = []
        for field in FINAL_SCHEMA:
            name = field.name
            if name == 'embedding':
                arrays.append(pa.FixedSizeListArray.from_arrays(pa.array(X.reshape(-1), type=pa.float32()), hidden_size))
            else:
                arrays.append(pa.array([row[name] for row in rows], type=field.type))
        return pa.Table.from_arrays(arrays, schema=FINAL_SCHEMA)

    def extract_batch_embeddings(batch_df, occ_id_start):
        batch_tokens = [str(value).split() for value in batch_df['context_window'].tolist()]
        batch_word_positions = batch_df['word_pos_window'].astype(int).tolist()
        raw_enc = tokenizer(batch_tokens, is_split_into_words=True, add_special_tokens=False, truncation=False, padding=False, return_attention_mask=False, verbose=False)
        prepared_input_ids = []
        wp_starts = []
        wp_ends = []
        crop_was_applied = []
        CLS_TOKEN_ID = tokenizer.cls_token_id
        SEP_TOKEN_ID = tokenizer.sep_token_id
        if CLS_TOKEN_ID is None:
            raise RuntimeError('El tokenizer no define cls_token_id.')
        if SEP_TOKEN_ID is None:
            raise RuntimeError('El tokenizer no define sep_token_id.')
        CLS_TOKEN_ID = int(CLS_TOKEN_ID)
        SEP_TOKEN_ID = int(SEP_TOKEN_ID)
        for batch_idx, word_position in enumerate(batch_word_positions):
            raw_ids = raw_enc['input_ids'][batch_idx]
            raw_word_ids = raw_enc.word_ids(batch_index=batch_idx)
            crop_info = compute_target_centered_wordpiece_crop(input_ids_no_special=raw_ids, word_ids=raw_word_ids, target_word_position=word_position)
            cropped_ids = crop_info['cropped_ids']
            model_ids = [CLS_TOKEN_ID] + [int(token_id) for token_id in cropped_ids] + [SEP_TOKEN_ID]
            if len(model_ids) > MAX_SEQ_LEN:
                raise RuntimeError(f'Secuencia BERT excede MAX_SEQ_LEN después del recorte centrado: {len(model_ids)} > {MAX_SEQ_LEN}')
            prepared_input_ids.append(model_ids)
            wp_starts.append(int(crop_info['wp_start']))
            wp_ends.append(int(crop_info['wp_end']))
            crop_was_applied.append(bool(crop_info['was_cropped']))
        padded = tokenizer.pad({'input_ids': prepared_input_ids}, padding=True, return_tensors='pt')
        input_ids = padded['input_ids'].to(DEVICE, non_blocking=True)
        attention_mask = padded['attention_mask'].to(DEVICE, non_blocking=True)
        batch_size = int(input_ids.shape[0])
        sequence_length = int(input_ids.shape[1])
        if sequence_length > MAX_SEQ_LEN:
            raise RuntimeError(f'El batch padded excede MAX_SEQ_LEN: {sequence_length} > {MAX_SEQ_LEN}')
        target_mask_np = np.zeros((batch_size, sequence_length), dtype=np.float32)
        for batch_idx, (wp_start, wp_end) in enumerate(zip(wp_starts, wp_ends)):
            if not 0 <= wp_start <= wp_end < sequence_length:
                raise RuntimeError(f'Posición WordPiece objetivo inválida: batch_idx={batch_idx}, wp_start={wp_start}, wp_end={wp_end}, sequence_length={sequence_length}')
            target_mask_np[batch_idx, wp_start:wp_end + 1] = 1.0
        captured_layer_outputs.clear()
        with torch.inference_mode():
            _ = model(input_ids=input_ids, attention_mask=attention_mask, return_dict=True)
            missing_layers = [layer_idx for layer_idx in resolved_layer_indices if layer_idx not in captured_layer_outputs]
            if missing_layers:
                raise RuntimeError(f'No se capturaron todas las capas BERT requeridas: {missing_layers}')
            token_reps = None
            for layer_idx in resolved_layer_indices:
                layer_tensor = captured_layer_outputs[layer_idx]
                if token_reps is None:
                    token_reps = layer_tensor
                else:
                    token_reps = token_reps + layer_tensor
            token_reps = token_reps / float(len(resolved_layer_indices))
            target_mask = torch.from_numpy(target_mask_np).to(DEVICE, dtype=token_reps.dtype, non_blocking=True)
            denominator = target_mask.sum(dim=1, keepdim=True)
            if torch.any(denominator <= 0):
                raise RuntimeError('Máscara WordPiece vacía.')
            pooled = (token_reps * target_mask.unsqueeze(-1)).sum(dim=1) / denominator
            if L2_NORMALIZE_EMBEDDINGS:
                pooled = torch.nn.functional.normalize(pooled, p=2, dim=1)
            X = pooled.detach().to('cpu', dtype=torch.float32).numpy()
        if X.ndim != 2:
            raise RuntimeError(f'Forma de embeddings inesperada: {X.shape}')
        if X.shape[1] != hidden_size:
            raise RuntimeError(f'Dimensión embedding incorrecta: {X.shape[1]} != {hidden_size}')
        if X.shape[0] != len(batch_df):
            raise RuntimeError('BERT cambió el número de observaciones dentro del batch.')
        if not np.isfinite(X).all():
            raise RuntimeError('BERT produjo NaN o Inf.')
        norms = np.linalg.norm(X, axis=1)
        if np.any(norms == 0):
            raise RuntimeError('BERT produjo un vector cero.')
        rows = []
        for local_idx, (_, source_row) in enumerate(batch_df.iterrows()):
            row = source_row.to_dict()
            row['occ_id'] = int(occ_id_start + local_idx)
            row['wp_start'] = int(wp_starts[local_idx])
            row['wp_end'] = int(wp_ends[local_idx])
            row.pop('sample_priority', None)
            rows.append(row)
        del target_mask
        del pooled
        del token_reps
        del input_ids
        del attention_mask
        del padded
        del raw_enc
        captured_layer_outputs.clear()
        return (rows, X)

    def extract_all_embeddings():
        if OUT_PARQUET_PATH.exists():
            print('El Parquet final ya existe. Se validará en las celdas siguientes.')
            parquet_file = pq.ParquetFile(OUT_PARQUET_PATH)
            return {'reused_existing_final_parquet': True, 'n_written': int(parquet_file.metadata.num_rows), 'output_path': str(OUT_PARQUET_PATH)}
        if OUT_PARQUET_PARTIAL_PATH.exists():
            print('Eliminando Parquet parcial de una ejecución anterior:', OUT_PARQUET_PARTIAL_PATH)
            raise RuntimeError('Existe un Parquet parcial; consérvalo y revisa su integridad antes de regenerar.')
        candidate_file = pq.ParquetFile(CANDIDATES_PATH)
        expected_rows = int(len(eligible_words) * OCCURRENCES_PER_WORD)
        if candidate_file.metadata.num_rows != expected_rows:
            raise RuntimeError(f'El candidate parquet no contiene el número esperado de filas: {candidate_file.metadata.num_rows} != {expected_rows}')
        writer = pq.ParquetWriter(OUT_PARQUET_PARTIAL_PATH, schema=FINAL_SCHEMA, compression='zstd', compression_level=1)
        output_rows_buffer = []
        output_embeddings_buffer = []
        occ_id = 0
        started = time.perf_counter()
        pbar = tqdm(total=expected_rows, desc='BERT occurrence embeddings', unit='occ')
        try:
            for batch in candidate_file.iter_batches(batch_size=BATCH_SIZE_OCCURRENCES):
                batch_df = batch.to_pandas()
                batch_rows, batch_embeddings = extract_batch_embeddings(batch_df, occ_id)
                if len(batch_rows) != len(batch_df):
                    raise RuntimeError('La extracción BERT cambió el número de filas.')
                output_rows_buffer.extend(batch_rows)
                output_embeddings_buffer.extend(batch_embeddings)
                occ_id += len(batch_rows)
                pbar.update(len(batch_rows))
                while len(output_rows_buffer) >= FINAL_ROW_GROUP_ROWS:
                    rows_chunk = output_rows_buffer[:FINAL_ROW_GROUP_ROWS]
                    embeddings_chunk = output_embeddings_buffer[:FINAL_ROW_GROUP_ROWS]
                    table = rows_embeddings_to_final_table(rows_chunk, embeddings_chunk)
                    writer.write_table(table, row_group_size=FINAL_ROW_GROUP_ROWS)
                    del output_rows_buffer[:FINAL_ROW_GROUP_ROWS]
                    del output_embeddings_buffer[:FINAL_ROW_GROUP_ROWS]
            if output_rows_buffer:
                table = rows_embeddings_to_final_table(output_rows_buffer, output_embeddings_buffer)
                writer.write_table(table, row_group_size=len(output_rows_buffer))
                output_rows_buffer.clear()
                output_embeddings_buffer.clear()
        finally:
            writer.close()
            candidate_file.close()
            pbar.close()
        if occ_id != expected_rows:
            raise RuntimeError(f'Se escribieron {occ_id} filas, esperadas {expected_rows}.')
        partial_pf = pq.ParquetFile(OUT_PARQUET_PARTIAL_PATH)
        try:
            partial_n_rows = int(partial_pf.metadata.num_rows)
        finally:
            partial_pf.close()
            del partial_pf
        gc.collect()
        if partial_n_rows != expected_rows:
            raise RuntimeError('El Parquet parcial cerrado no contiene el número esperado de filas.')
        os.replace(OUT_PARQUET_PARTIAL_PATH, OUT_PARQUET_PATH)
        return {'reused_existing_final_parquet': False, 'n_written': int(occ_id), 'n_words': int(len(eligible_words)), 'occurrences_per_word': int(OCCURRENCES_PER_WORD), 'embedding_dim': int(hidden_size), 'l2_normalization': bool(L2_NORMALIZE_EMBEDDINGS), 'selected_layers': list(SELECTED_LAYER_IDS), 'batch_size': int(BATCH_SIZE_OCCURRENCES), 'elapsed_seconds': float(time.perf_counter() - started), 'output_path': str(OUT_PARQUET_PATH)}
    extraction_report = extract_all_embeddings()
    print_copy_block('EXTRACCIÓN BERT', extraction_report)


## Construir índice


In [19]:
if GENERAR_DESDE_CORPUS:
    from pathlib import Path
    import math
    import pandas as pd
    import pyarrow.parquet as pq
    OUT_PARQUET_PATH = OUT_DIR / 'contextual_occurrences_definitive.parquet'
    FINAL_ELIGIBLE_VOCAB_PATH = OUT_DIR / 'eligible_vocabulary_min100.csv'
    ROW_GROUP_INDEX_PATH = OUT_DIR / 'word_row_group_index.csv'
    OCCURRENCES_PER_WORD = 100
    FINAL_ROW_GROUP_ROWS = 1000
    EXPECTED_N_WORDS = 103029
    EXPECTED_N_ROWS = EXPECTED_N_WORDS * OCCURRENCES_PER_WORD
    if not OUT_PARQUET_PATH.exists():
        raise FileNotFoundError(f'No existe el Parquet contextual definitivo:\n{OUT_PARQUET_PATH}')
    if not FINAL_ELIGIBLE_VOCAB_PATH.exists():
        raise FileNotFoundError(f'No existe el vocabulario definitivo:\n{FINAL_ELIGIBLE_VOCAB_PATH}')
    eligible_vocab_df = pd.read_csv(FINAL_ELIGIBLE_VOCAB_PATH, keep_default_na=False, na_filter=False)
    if 'word' not in eligible_vocab_df.columns:
        raise RuntimeError("eligible_vocabulary_min100.csv no contiene columna 'word'.")
    eligible_vocab_df['word'] = eligible_vocab_df['word'].astype(str).str.lower().str.strip()
    eligible_vocab_df = eligible_vocab_df.drop_duplicates(subset=['word']).sort_values('word').reset_index(drop=True)
    eligible_words = eligible_vocab_df['word'].tolist()
    if len(eligible_words) != EXPECTED_N_WORDS:
        raise RuntimeError(f'Número inesperado de palabras en eligible_vocabulary_min100.csv: {len(eligible_words):,} != {EXPECTED_N_WORDS:,}')
    if 'nan' in eligible_words:
        raise RuntimeError("La palabra 'nan' reapareció en el vocabulario definitivo.")
    if len(set(eligible_words)) != len(eligible_words):
        raise RuntimeError('El vocabulario contiene duplicados.')
    print('Vocabulario recuperado:', f'{len(eligible_words):,}')
    final_pf = pq.ParquetFile(OUT_PARQUET_PATH)
    try:
        n_rows_parquet = int(final_pf.metadata.num_rows)
        n_row_groups_actual = int(final_pf.metadata.num_row_groups)
        if n_rows_parquet != EXPECTED_N_ROWS:
            raise RuntimeError(f'Número de filas inesperado en el Parquet: {n_rows_parquet:,} != {EXPECTED_N_ROWS:,}')
        row_group_sizes = [int(final_pf.metadata.row_group(row_group_id).num_rows) for row_group_id in range(n_row_groups_actual)]
        expected_n_row_groups = int(math.ceil(EXPECTED_N_ROWS / FINAL_ROW_GROUP_ROWS))
        if n_row_groups_actual != expected_n_row_groups:
            raise RuntimeError(f'El número de row groups del Parquet no coincide con el diseño: {n_row_groups_actual:,} != {expected_n_row_groups:,}')
        if any((size != FINAL_ROW_GROUP_ROWS for size in row_group_sizes[:-1])):
            bad = [(idx, size) for idx, size in enumerate(row_group_sizes[:-1]) if size != FINAL_ROW_GROUP_ROWS]
            raise RuntimeError(f'Hay row groups intermedios con tamaño distinto de 1000. Ejemplos: {bad[:20]}')
        expected_last_row_group_size = int(EXPECTED_N_ROWS - (expected_n_row_groups - 1) * FINAL_ROW_GROUP_ROWS)
        if row_group_sizes[-1] != expected_last_row_group_size:
            raise RuntimeError(f'Tamaño inesperado del último row group: {row_group_sizes[-1]} != {expected_last_row_group_size}')
    finally:
        final_pf.close()
        del final_pf
    row_group_index_rows = []
    for word_index, word in enumerate(eligible_words):
        row_start = int(word_index * OCCURRENCES_PER_WORD)
        row_end = int(row_start + OCCURRENCES_PER_WORD)
        row_group_id = int(row_start // FINAL_ROW_GROUP_ROWS)
        row_group_index_rows.append({'word': word, 'word_index': int(word_index), 'n_occurrences': int(OCCURRENCES_PER_WORD), 'row_start': row_start, 'row_end_exclusive': row_end, 'row_group_id': row_group_id})
    word_row_group_index_df = pd.DataFrame(row_group_index_rows)
    if len(word_row_group_index_df) != EXPECTED_N_WORDS:
        raise RuntimeError(f'El índice no contiene exactamente {EXPECTED_N_WORDS:,} palabras.')
    if (word_row_group_index_df['n_occurrences'] != OCCURRENCES_PER_WORD).any():
        raise RuntimeError('El índice contiene una palabra con n_occurrences != 100.')
    if word_row_group_index_df['row_group_id'].max() >= expected_n_row_groups:
        raise RuntimeError('El índice referencia un row group inexistente.')
    word_row_group_index_df.to_csv(ROW_GROUP_INDEX_PATH, index=False, encoding='utf-8')
    index_report = {'parquet_path': str(OUT_PARQUET_PATH), 'row_group_index_path': str(ROW_GROUP_INDEX_PATH), 'n_words': int(EXPECTED_N_WORDS), 'occurrences_per_word': int(OCCURRENCES_PER_WORD), 'n_rows': int(EXPECTED_N_ROWS), 'row_group_rows': int(FINAL_ROW_GROUP_ROWS), 'n_row_groups': int(expected_n_row_groups), 'last_row_group_rows': int(expected_last_row_group_size), 'first_word': str(eligible_words[0]), 'last_word': str(eligible_words[-1]), 'nan_absent': bool('nan' not in eligible_words), 'index_valid': True}
    print_copy_block('ÍNDICE WORD ROW-GROUP', index_report)
    display(word_row_group_index_df.head(20))


## Validación completa


In [20]:
if GENERAR_DESDE_CORPUS:
    from pathlib import Path
    from collections import Counter
    import json
    import math
    import time
    import numpy as np
    import pandas as pd
    import pyarrow.parquet as pq
    OUT_PARQUET_PATH = OUT_DIR / 'contextual_occurrences_definitive.parquet'
    FINAL_ELIGIBLE_VOCAB_PATH = OUT_DIR / 'eligible_vocabulary_min100.csv'
    ROW_GROUP_INDEX_PATH = OUT_DIR / 'word_row_group_index.csv'
    VALIDATION_REPORT_PATH = OUT_DIR / 'validation_report.json'
    WORD_COUNTS_PATH = OUT_DIR / 'word_counts.csv'
    CATEGORY_COVERAGE_PATH = OUT_DIR / 'category_words_coverage.csv'
    OCCURRENCES_PER_WORD = 100
    EXPECTED_N_WORDS = 103029
    EXPECTED_N_ROWS = EXPECTED_N_WORDS * OCCURRENCES_PER_WORD
    EXPECTED_EMBEDDING_DIM = 768
    EXPECTED_ROW_GROUP_ROWS = 1000
    CATEGORY_WORDS = ['apple', 'bicycle', 'bread', 'canoe', 'chair', 'jellyfish', 'screwdriver', 'skunk']
    for required_path in [OUT_PARQUET_PATH, FINAL_ELIGIBLE_VOCAB_PATH, ROW_GROUP_INDEX_PATH]:
        if not required_path.exists():
            raise FileNotFoundError(f'No existe el archivo requerido:\n{required_path}')
    eligible_vocab_df = pd.read_csv(FINAL_ELIGIBLE_VOCAB_PATH, keep_default_na=False, na_filter=False)
    if 'word' not in eligible_vocab_df.columns:
        raise RuntimeError("eligible_vocabulary_min100.csv no contiene columna 'word'.")
    eligible_vocab_df['word'] = eligible_vocab_df['word'].astype(str).str.lower().str.strip()
    eligible_vocab_df = eligible_vocab_df.drop_duplicates(subset=['word']).sort_values('word').reset_index(drop=True)
    eligible_words = eligible_vocab_df['word'].tolist()
    eligible_word_set = set(eligible_words)
    if len(eligible_words) != EXPECTED_N_WORDS:
        raise RuntimeError(f'Número inesperado de palabras elegibles: {len(eligible_words):,} != {EXPECTED_N_WORDS:,}')
    if 'nan' in eligible_word_set:
        raise RuntimeError("'nan' reapareció en el vocabulario definitivo.")
    missing_category_words = sorted(set(CATEGORY_WORDS) - eligible_word_set)
    if missing_category_words:
        raise RuntimeError(f'Faltan CategoryWords en el vocabulario: {missing_category_words}')
    pf = pq.ParquetFile(OUT_PARQUET_PATH)
    try:
        n_rows = int(pf.metadata.num_rows)
        n_row_groups = int(pf.metadata.num_row_groups)
        schema_arrow = pf.schema_arrow
        if n_rows != EXPECTED_N_ROWS:
            raise RuntimeError(f'Número incorrecto de filas: {n_rows:,} != {EXPECTED_N_ROWS:,}')
        required_columns = ['occ_id', 'candidate_id', 'word', 'is_target', 'word_role', 'true_class', 'class_label', 'is_exp2_word', 'selection_stage', 'source_name', 'source_split', 'source_row_idx', 'source_doc_key', 'doc_split', 'word_pos_original', 'word_pos_window', 'wp_start', 'wp_end', 'n_tokens_window', 'context_window', 'left_context', 'right_context', 'text', 'embedding']
        missing_columns = [column for column in required_columns if column not in schema_arrow.names]
        if missing_columns:
            raise RuntimeError(f'Faltan columnas en el Parquet: {missing_columns}')
        word_counts = Counter()
        previous_word = None
        word_order_violations = 0
        n_word_rows_checked = 0
        started_words = time.perf_counter()
        for batch in pf.iter_batches(batch_size=200000, columns=['word']):
            words = batch.column(0).to_pylist()
            word_counts.update(words)
            for word in words:
                if previous_word is not None and word < previous_word:
                    word_order_violations += 1
                previous_word = word
            n_word_rows_checked += len(words)
        word_scan_seconds = float(time.perf_counter() - started_words)
        if n_word_rows_checked != EXPECTED_N_ROWS:
            raise RuntimeError('No se recorrieron todas las filas durante auditoría de words.')
        if word_order_violations != 0:
            raise RuntimeError(f'El Parquet no está ordenado por word. Violaciones={word_order_violations}')
        if len(word_counts) != EXPECTED_N_WORDS:
            raise RuntimeError(f'Número incorrecto de palabras únicas: {len(word_counts):,} != {EXPECTED_N_WORDS:,}')
        if set(word_counts) != eligible_word_set:
            missing_from_parquet = sorted(eligible_word_set - set(word_counts))
            unexpected_in_parquet = sorted(set(word_counts) - eligible_word_set)
            raise RuntimeError(f'El vocabulario del Parquet no coincide con eligible_vocabulary_min100.csv. Missing={missing_from_parquet[:20]}, Unexpected={unexpected_in_parquet[:20]}')
        bad_word_counts = {word: count for word, count in word_counts.items() if count != OCCURRENCES_PER_WORD}
        if bad_word_counts:
            raise RuntimeError(f'Existen palabras con conteo distinto de 100. Ejemplos={list(bad_word_counts.items())[:20]}')
        category_coverage_df = pd.DataFrame([{'category_word': word, 'n_occurrences': int(word_counts[word]), 'exactly_100': bool(word_counts[word] == OCCURRENCES_PER_WORD)} for word in CATEGORY_WORDS])
        if not category_coverage_df['exactly_100'].all():
            raise RuntimeError('Alguna CategoryWord no tiene exactamente 100 ocurrencias.')
        n_embedding_rows_checked = 0
        n_non_finite_embeddings = 0
        n_zero_embeddings = 0
        embedding_dim = None
        global_norm_min = float('inf')
        global_norm_max = float('-inf')
        norm_sum = 0.0
        norm_sq_sum = 0.0
        started_embeddings = time.perf_counter()
        for batch in pf.iter_batches(batch_size=4096, columns=['embedding']):
            embedding_array = batch.column(0)
            X = embedding_array.values.to_numpy(zero_copy_only=False).reshape(len(batch), -1).astype(np.float32, copy=False)
            if embedding_dim is None:
                embedding_dim = int(X.shape[1])
            if X.shape[1] != EXPECTED_EMBEDDING_DIM:
                raise RuntimeError(f'Dimensión embedding incorrecta: {X.shape[1]} != {EXPECTED_EMBEDDING_DIM}')
            finite_mask = np.isfinite(X).all(axis=1)
            n_non_finite_embeddings += int((~finite_mask).sum())
            zero_mask = np.all(X == 0, axis=1)
            n_zero_embeddings += int(zero_mask.sum())
            norms = np.linalg.norm(X, axis=1)
            global_norm_min = min(global_norm_min, float(norms.min()))
            global_norm_max = max(global_norm_max, float(norms.max()))
            norm_sum += float(norms.astype(np.float64).sum())
            norm_sq_sum += float(np.square(norms.astype(np.float64)).sum())
            n_embedding_rows_checked += int(X.shape[0])
        embedding_scan_seconds = float(time.perf_counter() - started_embeddings)
        if n_embedding_rows_checked != EXPECTED_N_ROWS:
            raise RuntimeError(f'No se auditaron todos los embeddings: {n_embedding_rows_checked:,} != {EXPECTED_N_ROWS:,}')
        if n_non_finite_embeddings != 0:
            raise RuntimeError(f'Se detectaron embeddings con NaN/Inf: {n_non_finite_embeddings}')
        if n_zero_embeddings != 0:
            raise RuntimeError(f'Se detectaron embeddings cero: {n_zero_embeddings}')
        norm_mean = norm_sum / n_embedding_rows_checked
        norm_variance = norm_sq_sum / n_embedding_rows_checked - norm_mean ** 2
        norm_std = math.sqrt(max(0.0, norm_variance))
    finally:
        pf.close()
        del pf
    row_group_index_df = pd.read_csv(ROW_GROUP_INDEX_PATH, keep_default_na=False, na_filter=False)
    required_index_columns = ['word', 'word_index', 'n_occurrences', 'row_start', 'row_end_exclusive', 'row_group_id']
    missing_index_columns = [column for column in required_index_columns if column not in row_group_index_df.columns]
    if missing_index_columns:
        raise RuntimeError(f'Faltan columnas en word_row_group_index.csv: {missing_index_columns}')
    if len(row_group_index_df) != EXPECTED_N_WORDS:
        raise RuntimeError('Número incorrecto de filas en word_row_group_index.csv.')
    if row_group_index_df['word'].astype(str).tolist() != eligible_words:
        raise RuntimeError('El orden de words del índice no coincide con el vocabulario definitivo.')
    if row_group_index_df['n_occurrences'].astype(int).ne(OCCURRENCES_PER_WORD).any():
        raise RuntimeError('El índice contiene n_occurrences != 100.')
    word_counts_df = pd.DataFrame({'word': eligible_words, 'n_occurrences': [int(word_counts[word]) for word in eligible_words]})
    word_counts_df.to_csv(WORD_COUNTS_PATH, index=False, encoding='utf-8')
    category_coverage_df.to_csv(CATEGORY_COVERAGE_PATH, index=False, encoding='utf-8')
    validation_report = {'status': 'VALID', 'parquet_path': str(OUT_PARQUET_PATH), 'parquet_size_gib': float(OUT_PARQUET_PATH.stat().st_size / 1024 ** 3), 'n_rows': int(EXPECTED_N_ROWS), 'n_unique_words': int(EXPECTED_N_WORDS), 'occurrences_per_word': int(OCCURRENCES_PER_WORD), 'all_words_exactly_100': bool(not bad_word_counts), 'all_8_category_words_exactly_100': bool(category_coverage_df['exactly_100'].all()), 'category_words': CATEGORY_WORDS, 'nan_absent': bool('nan' not in eligible_word_set), 'word_order_violations': int(word_order_violations), 'embedding_dim': int(embedding_dim), 'embedding_dtype': 'float32', 'l2_normalized': False, 'n_embedding_rows_checked': int(n_embedding_rows_checked), 'n_non_finite_embeddings': int(n_non_finite_embeddings), 'n_zero_embeddings': int(n_zero_embeddings), 'embedding_norm_min': float(global_norm_min), 'embedding_norm_max': float(global_norm_max), 'embedding_norm_mean': float(norm_mean), 'embedding_norm_std': float(norm_std), 'n_row_groups': int(n_row_groups), 'row_group_rows_target': int(EXPECTED_ROW_GROUP_ROWS), 'row_group_index_valid': True, 'word_scan_seconds': float(word_scan_seconds), 'embedding_scan_seconds': float(embedding_scan_seconds)}
    with open(VALIDATION_REPORT_PATH, 'w', encoding='utf-8') as handle:
        json.dump(validation_report, handle, ensure_ascii=False, indent=2, sort_keys=True)
    print('\n' + '=' * 90)
    print('PARA COPIAR Y PEGAR — VALIDACIÓN FINAL DATASET CONTEXTUAL')
    print('=' * 90)
    print(json.dumps(validation_report, ensure_ascii=False, indent=2, sort_keys=True))
    print('=' * 90)
    print('FIN — VALIDACIÓN FINAL DATASET CONTEXTUAL')
    print('=' * 90)


## Metadatos finales


In [21]:
if GENERAR_DESDE_CORPUS:
    from pathlib import Path
    import json
    import pandas as pd
    import pyarrow.parquet as pq
    OUT_PARQUET_PATH = OUT_DIR / 'contextual_occurrences_definitive.parquet'
    FINAL_ELIGIBLE_VOCAB_PATH = OUT_DIR / 'eligible_vocabulary_min100.csv'
    ROW_GROUP_INDEX_PATH = OUT_DIR / 'word_row_group_index.csv'
    SELECTION_REPORT_PATH = OUT_DIR / 'selection_report.json'
    VALIDATION_REPORT_PATH = OUT_DIR / 'validation_report.json'
    TOKENIZER_ALIGNMENT_AUDIT_PATH = OUT_DIR / 'tokenizer_alignment_audit.json'
    META_PATH = OUT_DIR / 'contextual_dataset_meta.json'
    GENERATION_CONFIG_PATH = OUT_DIR / 'generation_config.json'
    MODEL_NAME = 'bert-base-uncased'
    EMBEDDING_DIM = 768
    SELECTED_LAYER_IDS = [-4, -3, -2, -1]
    LAYER_AGGREGATION = 'mean'
    TARGET_WORDPIECE_AGGREGATION = 'mean'
    L2_NORMALIZE_EMBEDDINGS = False
    OCCURRENCES_PER_WORD = 100
    EXPECTED_N_WORDS = 103029
    EXPECTED_N_ROWS = 10302900
    WINDOW_TOKENS_EACH_SIDE = 40
    MAX_SEQ_LEN = 128
    N_SPECIAL_TOKENS = 2
    MAX_CONTENT_WORDPIECES = MAX_SEQ_LEN - N_SPECIAL_TOKENS
    FINAL_ROW_GROUP_ROWS = 1000
    CONTEXTUAL_DATASET_SCHEMA_VERSION = 'ctx_v3_commonpool_equal100_rawbert'
    CONFIG_NAME = 'ctx_v3_bert_base_commonpool_equal100'
    EXPECTED_SELECTION_SHA256 = 'e130061a093ff98c3265cda041a3a86e77c6204c3509e0aa0a56405af87af087'
    CATEGORY_WORDS = ['apple', 'bicycle', 'bread', 'canoe', 'chair', 'jellyfish', 'screwdriver', 'skunk']

    def load_json_required(path):
        if not path.exists():
            raise FileNotFoundError(f'No existe el JSON requerido:\n{path}')
        with open(path, 'r', encoding='utf-8') as handle:
            return json.load(handle)

    def save_json(path, payload):
        with open(path, 'w', encoding='utf-8') as handle:
            json.dump(payload, handle, ensure_ascii=False, indent=2, sort_keys=True)
    selection_report = load_json_required(SELECTION_REPORT_PATH)
    validation_report = load_json_required(VALIDATION_REPORT_PATH)
    alignment_audit = load_json_required(TOKENIZER_ALIGNMENT_AUDIT_PATH)
    selection_sha256 = selection_report.get('selection_sha256_final') or selection_report.get('selection_sha256')
    if selection_sha256 is None:
        raise RuntimeError('selection_report.json no contiene selection_sha256_final ni selection_sha256.')
    if selection_sha256 != EXPECTED_SELECTION_SHA256:
        raise RuntimeError(f'Fingerprint de selección inesperado:\n{selection_sha256}\nEsperado:\n{EXPECTED_SELECTION_SHA256}')
    if validation_report.get('status') != 'VALID':
        raise RuntimeError("validation_report.json no tiene status='VALID'.")
    if int(validation_report['n_rows']) != EXPECTED_N_ROWS:
        raise RuntimeError('Número final de filas inesperado.')
    if int(validation_report['n_unique_words']) != EXPECTED_N_WORDS:
        raise RuntimeError('Número final de palabras inesperado.')
    if not bool(validation_report['all_words_exactly_100']):
        raise RuntimeError('No todas las palabras tienen exactamente 100 ocurrencias.')
    if not bool(validation_report['all_8_category_words_exactly_100']):
        raise RuntimeError('Las CategoryWords no satisfacen la cobertura definitiva.')
    if int(validation_report['n_non_finite_embeddings']) != 0:
        raise RuntimeError('El reporte registra embeddings no finitos.')
    if int(validation_report['n_zero_embeddings']) != 0:
        raise RuntimeError('El reporte registra embeddings cero.')
    if not bool(alignment_audit.get('all_rows_alignable', False)):
        raise RuntimeError('La auditoría WordPiece no indica all_rows_alignable=True.')
    if int(alignment_audit.get('n_centered_crop_failures', 0)) != 0:
        raise RuntimeError('La auditoría registra fallos del recorte WordPiece centrado.')
    if int(alignment_audit.get('n_target_missing_before_crop', 0)) != 0:
        raise RuntimeError('La auditoría registra targets ausentes antes del recorte.')
    if int(alignment_audit.get('n_target_too_long', 0)) != 0:
        raise RuntimeError('La auditoría registra targets mayores que el presupuesto WordPiece.')
    eligible_vocab_df = pd.read_csv(FINAL_ELIGIBLE_VOCAB_PATH, keep_default_na=False, na_filter=False)
    eligible_vocab_df['word'] = eligible_vocab_df['word'].astype(str).str.lower().str.strip()
    eligible_vocab_df = eligible_vocab_df.drop_duplicates(subset=['word']).sort_values('word').reset_index(drop=True)
    eligible_words = eligible_vocab_df['word'].tolist()
    if len(eligible_words) != EXPECTED_N_WORDS:
        raise RuntimeError('eligible_vocabulary_min100.csv no contiene exactamente 103029 palabras.')
    if 'nan' in eligible_words:
        raise RuntimeError("'nan' reapareció en el vocabulario.")
    if 'null' not in eligible_words:
        raise RuntimeError("El literal válido 'null' desapareció inesperadamente del vocabulario.")
    row_group_index_df = pd.read_csv(ROW_GROUP_INDEX_PATH, keep_default_na=False, na_filter=False)
    if len(row_group_index_df) != EXPECTED_N_WORDS:
        raise RuntimeError('word_row_group_index.csv tiene un número inesperado de palabras.')
    pf = pq.ParquetFile(OUT_PARQUET_PATH)
    try:
        parquet_n_rows = int(pf.metadata.num_rows)
        parquet_n_row_groups = int(pf.metadata.num_row_groups)
    finally:
        pf.close()
        del pf
    if parquet_n_rows != EXPECTED_N_ROWS:
        raise RuntimeError('El Parquet definitivo no contiene 10.302.900 filas.')
    generation_config = {'config_name': CONFIG_NAME, 'schema_version': CONTEXTUAL_DATASET_SCHEMA_VERSION, 'model': {'name': MODEL_NAME, 'architecture': 'BERT-base uncased', 'embedding_dim': EMBEDDING_DIM, 'selected_layers': SELECTED_LAYER_IDS, 'layer_aggregation': LAYER_AGGREGATION, 'target_wordpiece_aggregation': TARGET_WORDPIECE_AGGREGATION, 'l2_normalized': L2_NORMALIZE_EMBEDDINGS, 'embedding_dtype': 'float32'}, 'context': {'lexical_tokens_each_side': WINDOW_TOKENS_EACH_SIDE, 'context_window_definition': 'target ±40 lexical whitespace tokens', 'stopwords_retained_inside_context': True, 'context_is_selection_unit': True}, 'bert_input_policy': {'policy': 'target_centered_wordpiece_crop', 'max_seq_len': MAX_SEQ_LEN, 'n_special_tokens': N_SPECIAL_TOKENS, 'max_content_wordpieces': MAX_CONTENT_WORDPIECES, 'special_tokens': ['[CLS]', '[SEP]'], 'preserve_all_target_wordpieces': True, 'standard_right_truncation_used': False, 'crop_applied_before_model_forward': True, 'target_positions_recorded_as': ['wp_start', 'wp_end']}, 'occurrence_selection': {'occurrences_per_word': OCCURRENCES_PER_WORD, 'selection_method': 'blake2b_128_keep_100_smallest_unique_word_context_hashes', 'selection_personalization': 'ctx100-v1', 'selection_sha256': selection_sha256, 'same_sampling_policy_for_all_words': True, 'category_words_sampling_privilege': False}, 'corpus_pool': {'policy': 'common_pool_same_for_every_word', 'sources': {'wikitext_103_raw': {'dataset': 'wikitext-103-raw', 'records_policy': 'all available records'}, 'wikimedia_wikipedia_20231101_en_common': {'dataset': 'wikimedia/wikipedia', 'subset': '20231101.en', 'records_policy': 'first 800000 train records', 'max_records': 800000}}, 'source_documents_scanned': selection_report.get('source_documents_scanned')}, 'vocabulary': {'n_words': EXPECTED_N_WORDS, 'minimum_unique_usable_occurrences': OCCURRENCES_PER_WORD, 'automatic_candidate_contract': {'regex': '[a-z]+', 'minimum_length': 3, 'english_stopwords_excluded_as_candidates': True, 'custom_bad_words_excluded': True, 'pos_filter': None}, 'literal_nan_excluded': True, 'literal_null_retained': True, 'conceptual_lexicon_filter_applied': False, 'conceptual_lexicon_filter_note': 'A downstream conceptual vocabulary may be constructed for prototype/neighbor analyses without modifying this master occurrence-level dataset.'}, 'final_dataset': {'n_words': EXPECTED_N_WORDS, 'occurrences_per_word': OCCURRENCES_PER_WORD, 'n_rows': EXPECTED_N_ROWS, 'ordered_by': ['word ASC', 'sample_priority ASC'], 'row_group_target_rows': FINAL_ROW_GROUP_ROWS, 'n_row_groups': parquet_n_row_groups, 'occurrence_level': True}, 'downstream_static_prototype_policy': {'prototype_definition': 'arithmetic mean of the 100 raw contextual occurrence embeddings for each word', 'formula': 'p_w = (1/100) * sum_i x_raw[w,i]', 'l2_before_mean': False, 'l2_after_mean': False, 'prototype_already_stored_in_master_dataset': False}}
    dataset_meta = {'status': 'DEFINITIVE_VALIDATED', 'schema_version': CONTEXTUAL_DATASET_SCHEMA_VERSION, 'config_name': CONFIG_NAME, 'parquet_path': str(OUT_PARQUET_PATH), 'eligible_vocabulary_path': str(FINAL_ELIGIBLE_VOCAB_PATH), 'word_row_group_index_path': str(ROW_GROUP_INDEX_PATH), 'validation_report_path': str(VALIDATION_REPORT_PATH), 'tokenizer_alignment_audit_path': str(TOKENIZER_ALIGNMENT_AUDIT_PATH), 'generation_config_path': str(GENERATION_CONFIG_PATH), 'n_words': EXPECTED_N_WORDS, 'occurrences_per_word': OCCURRENCES_PER_WORD, 'n_rows': EXPECTED_N_ROWS, 'embedding_dim': EMBEDDING_DIM, 'embedding_dtype': 'float32', 'l2_normalized': False, 'model': MODEL_NAME, 'selected_layers': SELECTED_LAYER_IDS, 'context_tokens_each_side': WINDOW_TOKENS_EACH_SIDE, 'max_seq_len': MAX_SEQ_LEN, 'wordpiece_input_policy': 'target_centered_wordpiece_crop', 'max_content_wordpieces': MAX_CONTENT_WORDPIECES, 'preserve_all_target_wordpieces': True, 'selection_sha256': selection_sha256, 'all_words_exactly_100': True, 'all_8_category_words_exactly_100': True, 'all_embeddings_finite': True, 'zero_embeddings': 0, 'word_order_violations': 0, 'row_group_index_valid': True, 'conceptual_filter_applied': False, 'master_dataset_should_remain_immutable': True}
    save_json(GENERATION_CONFIG_PATH, generation_config)
    save_json(META_PATH, dataset_meta)
    if not GENERATION_CONFIG_PATH.exists():
        raise RuntimeError('No se escribió generation_config.json.')
    if not META_PATH.exists():
        raise RuntimeError('No se escribió contextual_dataset_meta.json.')
    metadata_report = {'status': 'METADATA_FINALIZED', 'dataset_status': dataset_meta['status'], 'n_words': EXPECTED_N_WORDS, 'n_rows': EXPECTED_N_ROWS, 'occurrences_per_word': OCCURRENCES_PER_WORD, 'embedding_dim': EMBEDDING_DIM, 'l2_normalized': False, 'wordpiece_input_policy': 'target_centered_wordpiece_crop', 'max_seq_len': MAX_SEQ_LEN, 'max_content_wordpieces': MAX_CONTENT_WORDPIECES, 'selection_sha256': selection_sha256, 'generation_config_path': str(GENERATION_CONFIG_PATH), 'dataset_meta_path': str(META_PATH)}
    print('\n' + '=' * 90)
    print('PARA COPIAR Y PEGAR — METADATOS DEFINITIVOS DATASET CONTEXTUAL')
    print('=' * 90)
    print(json.dumps(metadata_report, ensure_ascii=False, indent=2, sort_keys=True))
    print('=' * 90)
    print('FIN — METADATOS DEFINITIVOS DATASET CONTEXTUAL')
    print('=' * 90)


## Verificación final de los prototipos

La salida regenerada solo se acepta si reproduce exactamente el vocabulario y los valores definitivos.


In [22]:
if GENERAR_DESDE_CORPUS:
    df_regenerado = ctx.reconstruir_prototipos(BASE, parquet=OUT_PARQUET_PATH,
        output=BASE/'SalidasRegeneracion/prototipos_desde_BERT.npz')
    print(ctx.verificar_prototipos(df_regenerado))
